# Segment family power check, no arm-by-outcome splits

The design doc names the pre-registered segments (newbie, channel, zip_code, recency tertiles) and BH correction, but not which comparisons run inside each segment level. Candidate families:

- **A.** Mens vs No E-Mail and Womens vs No E-Mail in each of the 11 segment levels, 22 tests
- **B.** All three comparisons (adding Mens vs Womens) in each level, 33 tests
- **C.** Interaction tests (does the lift differ across a segment's levels), 8 tests

The data can't pick the family, that is a decision-logic question. What it can show is the power cost of each option, which this notebook computes. It stays on the right side of the pre-registration line:
- arm sizes within each level are counts of `segment`, no outcome column involved
- baseline conversion per level is pooled across all three arms, never split by arm

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import brentq
from load import load_hillstrom

df = load_hillstrom()

df["recency_bin"] = pd.cut(df["recency"], bins=[0, 3, 8, 12], labels=["<=3", "4-8", "9+"], include_lowest=True)
df["newbie"] = df["newbie"].astype(str)

segment_vars = ["newbie", "channel", "zip_code", "recency_bin"]
alpha = 0.05
power = 0.90
mde_absolute = 0.005

print(f"rows: {len(df)}")
print(f"segment variables: {segment_vars}")
print(f"locked alpha {alpha}, power {power}, MDE {mde_absolute:.2%}")

rows: 64000
segment variables: ['newbie', 'channel', 'zip_code', 'recency_bin']
locked alpha 0.05, power 0.9, MDE 0.50%


## Step 1. Customers per arm within each segment level

Counts only. A segment comparison like Mens vs No E-Mail inside Multichannel can only use the Multichannel customers in those two arms, so these counts set the sample size for every segment-level test.

In [2]:
rows = []
for var in segment_vars:
    ct = pd.crosstab(df[var], df["segment"])
    for level, r in ct.iterrows():
        rows.append({"variable": var, "level": str(level), **r.to_dict(), "level_total": r.sum()})
level_counts = pd.DataFrame(rows)[["variable", "level", "Mens E-Mail", "Womens E-Mail", "No E-Mail", "level_total"]]
print(f"number of segment levels: {len(level_counts)}")
level_counts

number of segment levels: 11


,variable,level,Mens E-Mail,Womens E-Mail,No E-Mail,level_total
0,newbie,0,10621,10624,10611,31856
1,newbie,1,10686,10763,10695,32144
2,channel,Multichannel,2577,2579,2606,7762
3,channel,Phone,9240,9454,9327,28021
4,channel,Web,9490,9354,9373,28217
5,zip_code,Rural,3243,3181,3139,9563
6,zip_code,Surburban,9501,9650,9625,28776
7,zip_code,Urban,8563,8556,8542,25661
8,recency_bin,<=3,7469,7486,7438,22393
9,recency_bin,4-8,7157,7269,7339,21765


## Step 2. Pooled baseline conversion per level

Conversion rate in each level, pooled across all three arms. This mixes in any treatment effect from the two email arms, so it sits slightly above the true no-email baseline. For rates this far below 50%, a higher p means a larger variance p(1-p), so using it makes the power numbers below slightly conservative, not optimistic.

In [3]:
pooled_rates = []
for var in segment_vars:
    rates = df.groupby(var, observed=True)["conversion"].mean()
    for level, rate in rates.items():
        pooled_rates.append({"variable": var, "level": str(level), "pooled_conversion": rate})
pooled_rates = pd.DataFrame(pooled_rates)

seg = level_counts.merge(pooled_rates, on=["variable", "level"])
seg[["variable", "level", "level_total", "pooled_conversion"]].style.format({"pooled_conversion": "{:.4%}"})

,variable,level,level_total,pooled_conversion
0,newbie,0,31856,1.0108%
1,newbie,1,32144,0.7964%
2,channel,Multichannel,7762,1.2626%
3,channel,Phone,28021,0.7744%
4,channel,Web,28217,0.9321%
5,zip_code,Rural,9563,1.1189%
6,zip_code,Surburban,28776,0.8306%
7,zip_code,Urban,25661,0.9041%
8,recency_bin,<=3,22393,1.2370%
9,recency_bin,4-8,21765,0.7535%


## Step 3. The alpha range BH can apply

BH doesn't use one fixed threshold. It sorts the m p-values and compares the k-th smallest to (k/m) * alpha. So the bar a single test has to clear depends on the other p-values, it ranges between two extremes:

- loosest, alpha (the bar for the largest p-value when everything is significant)
- strictest, alpha / m (the bar for the smallest p-value when nothing else is, same as Bonferroni)

Real power under BH lands somewhere in that band. Computing the two-sided critical z at each end, for A (m = 2 comparisons x number of levels) and B (m = 3 comparisons x number of levels), with the level count taken from Step 1.

In [4]:
n_levels = len(level_counts)
m_A = 2 * n_levels
m_B = 3 * n_levels
print(f"segment levels: {n_levels}, tests in A: {m_A}, tests in B: {m_B}")

z_power = stats.norm.ppf(power)
scenarios = {
    "loosest (alpha)": alpha,
    "A strictest (alpha/m_A)": alpha / m_A,
    "B strictest (alpha/m_B)": alpha / m_B,
}
z_alpha = {name: stats.norm.ppf(1 - a / 2) for name, a in scenarios.items()}

print(f"z_power for {power:.0%} power: {z_power:.4f}")
for name, a in scenarios.items():
    print(f"{name}: alpha used {a:.5f}, z_(alpha/2) {z_alpha[name]:.4f}")

segment levels: 11, tests in A: 22, tests in B: 33
z_power for 90% power: 1.2816
loosest (alpha): alpha used 0.05000, z_(alpha/2) 1.9600
A strictest (alpha/m_A): alpha used 0.00227, z_(alpha/2) 3.0521
B strictest (alpha/m_B): alpha used 0.00152, z_(alpha/2) 3.1718


## Step 4. Smallest detectable lift per level

Same relationship as `00_power_and_design.ipynb`, generalized to unequal arm sizes since arms differ slightly inside each level:

delta = (z_(alpha/2) + z_power) * sqrt( p1(1-p1)/n1 + p2(1-p2)/n2 ), with p2 = p1 + delta

delta appears on both sides, so it is solved numerically. n1 is the No E-Mail count in the level (Mens vs Womens uses the Womens count), n2 the email arm's count. To keep one number per level, each comparison uses the smaller of its two arms.

In [5]:
def detectable_delta(p1, n1, n2, z_a):
    def f(delta):
        p2 = p1 + delta
        se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
        return (z_a + z_power) * se - delta
    return brentq(f, 1e-7, 0.5)

out = seg[["variable", "level", "pooled_conversion"]].copy()
out["n_email_vs_none"] = seg[["Mens E-Mail", "Womens E-Mail", "No E-Mail"]].min(axis=1)
for name in scenarios:
    out[f"mde_{name}"] = [
        detectable_delta(p, n, n, z_alpha[name]) for p, n in zip(out["pooled_conversion"], out["n_email_vs_none"])
    ]

fmt = {c: "{:.4%}" for c in out.columns if c.startswith("mde_")}
fmt["pooled_conversion"] = "{:.4%}"
out.style.format(fmt)

,variable,level,pooled_conversion,n_email_vs_none,mde_loosest (alpha),mde_A strictest (alpha/m_A),mde_B strictest (alpha/m_B)
0,newbie,0,1.0108%,10611,0.4960%,0.6874%,0.7092%
1,newbie,1,0.7964%,10686,0.4453%,0.6199%,0.6398%
2,channel,Multichannel,1.2626%,2577,1.2234%,1.7412%,1.8014%
3,channel,Phone,0.7744%,9240,0.4775%,0.6670%,0.6888%
4,channel,Web,0.9321%,9354,0.5136%,0.7145%,0.7374%
5,zip_code,Rural,1.1189%,3139,1.0376%,1.4741%,1.5248%
6,zip_code,Surburban,0.8306%,9501,0.4844%,0.6753%,0.6971%
7,zip_code,Urban,0.9041%,8542,0.5334%,0.7439%,0.7679%
8,recency_bin,<=3,1.2370%,7438,0.6599%,0.9168%,0.9461%
9,recency_bin,4-8,0.7535%,7157,0.5460%,0.7677%,0.7932%


## Step 5. Achieved power per level at the locked MDE (0.5pp)

The targeted-rollout tier of the decision rule needs a segment lift that is both significant and at least 0.5pp. So the question that matters is how likely each level is to catch a true 0.5pp lift. Solving the same relationship for power:

z_power_achieved = delta / sqrt( p1(1-p1)/n1 + p2(1-p2)/n2 ) - z_(alpha/2), then power = Phi(z_power_achieved)

In [6]:
def achieved_power(p1, n1, n2, z_a, delta=mde_absolute):
    p2 = p1 + delta
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    return stats.norm.cdf(delta / se - z_a)

pw = out[["variable", "level", "n_email_vs_none"]].copy()
for name in scenarios:
    pw[f"power_{name}"] = [
        achieved_power(p, n, n, z_alpha[name]) for p, n in zip(out["pooled_conversion"], out["n_email_vs_none"])
    ]
pw.style.format({c: "{:.2%}" for c in pw.columns if c.startswith("power_")})

,variable,level,n_email_vs_none,power_loosest (alpha),power_A strictest (alpha/m_A),power_B strictest (alpha/m_B)
0,newbie,0,10611,90.40%,58.43%,53.71%
1,newbie,1,10686,94.87%,70.56%,66.31%
2,channel,Multichannel,2577,31.24%,5.69%,4.45%
3,channel,Phone,9240,92.16%,62.71%,58.10%
4,channel,Web,9354,88.58%,54.48%,49.72%
5,zip_code,Rural,3139,39.94%,8.90%,7.12%
6,zip_code,Surburban,9501,91.53%,61.11%,56.46%
7,zip_code,Urban,8542,86.44%,50.33%,45.56%
8,recency_bin,<=3,7438,71.22%,29.73%,25.72%
9,recency_bin,4-8,7157,85.13%,48.00%,43.25%


## Step 6. What option C would cost, for scale

An interaction test asks whether the lift differs between levels. For the simplest case, a two-level segment (newbie), the estimate is a difference of two lifts, so its variance is the sum of four arm-level variances, roughly double that of a single lift. Computing the detectable difference in lift for newbie (Mens vs No E-Mail), using the loosest alpha, to show the scale. Three-level segments would need a multi-degree-of-freedom test and are not computed here.

In [7]:
nb_rows = seg[seg["variable"] == "newbie"].set_index("level")
z_a = z_alpha["loosest (alpha)"]

def interaction_detectable(p_a, p_b, n_a, n_b):
    def f(d):
        se = np.sqrt(2 * p_a * (1 - p_a) / n_a + 2 * p_b * (1 - p_b) / n_b)
        return (z_a + z_power) * se - d
    return brentq(f, 1e-7, 0.5)

n_a = nb_rows.loc["0", ["Mens E-Mail", "No E-Mail"]].min()
n_b = nb_rows.loc["1", ["Mens E-Mail", "No E-Mail"]].min()
p_a = nb_rows.loc["0", "pooled_conversion"]
p_b = nb_rows.loc["1", "pooled_conversion"]
d_int = interaction_detectable(p_a, p_b, n_a, n_b)

print(f"newbie=0: n per arm {n_a:,}, pooled conversion {p_a:.4%}")
print(f"newbie=1: n per arm {n_b:,}, pooled conversion {p_b:.4%}")
print(f"detectable difference in lift between newbie levels (alpha {alpha}, power {power:.0%}): {d_int*100:.4f} percentage points")

newbie=0: n per arm 10,611, pooled conversion 1.0108%
newbie=1: n per arm 10,686, pooled conversion 0.7964%
detectable difference in lift between newbie levels (alpha 0.05, power 90%): 0.5946 percentage points


## Step 7. Expected conversions per arm in each level (added 2026-10-19, before any segment outcome)

The segment tests use a Wald interval, which leans on a normal approximation. That approximation gets shaky when a cell has few conversions, so this checks how many each arm can be expected to have in each level. Still no segment outcome is split by arm.

Two versions:

- pooled, expected conversions = arm count in the level x the level's pooled conversion rate (Step 2)
- No E-Mail lower side, No E-Mail count in the level x the overall No E-Mail conversion rate. The overall No E-Mail rate was already computed in the primary readout (0.5726%), and it sits below the pooled rate, so this is the smaller and more cautious count for the arm with the fewest conversions. It uses only the overall arm rate, not any segment-level outcome

Expected non-conversions are far larger in every cell, so the conversion count is the side that matters. The Newcombe interval reported beside Wald is the real safeguard, these counts show where it is most likely to matter.


In [8]:
no_email_overall_rate = df.loc[df["segment"] == "No E-Mail", "conversion"].mean()
print(f"overall No E-Mail conversion rate (from the primary readout): {no_email_overall_rate:.4%}")

exp = seg[["variable", "level", "pooled_conversion"]].copy()
for arm in ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]:
    exp[f"expected_conv_{arm}"] = seg[arm] * seg["pooled_conversion"]
exp["expected_conv_No E-Mail_lower"] = seg["No E-Mail"] * no_email_overall_rate
exp["smallest_expected"] = exp[[c for c in exp.columns if c.startswith("expected_conv_")]].min(axis=1)
exp = exp.sort_values("smallest_expected").reset_index(drop=True)
exp.style.format({c: "{:.2f}" for c in exp.columns if c.startswith("expected_conv_") or c == "smallest_expected"} | {"pooled_conversion": "{:.4%}"})


overall No E-Mail conversion rate (from the primary readout): 0.5726%


,variable,level,pooled_conversion,expected_conv_Mens E-Mail,expected_conv_Womens E-Mail,expected_conv_No E-Mail,expected_conv_No E-Mail_lower,smallest_expected
0,channel,Multichannel,1.2626%,32.54,32.56,32.90,14.92,14.92
1,zip_code,Rural,1.1189%,36.29,35.59,35.12,17.97,17.97
2,recency_bin,9+,0.6905%,46.13,45.79,45.08,37.39,37.39
3,recency_bin,4-8,0.7535%,53.93,54.77,55.30,42.02,42.02
4,recency_bin,<=3,1.2370%,92.39,92.60,92.01,42.59,42.59
5,zip_code,Urban,0.9041%,77.42,77.35,77.23,48.91,48.91
6,channel,Phone,0.7744%,71.56,73.21,72.23,53.41,53.41
7,channel,Web,0.9321%,88.45,87.19,87.36,53.67,53.67
8,zip_code,Surburban,0.8306%,78.91,80.15,79.94,55.11,55.11
9,newbie,0,1.0108%,107.36,107.39,107.26,60.76,60.76


## Save results

In [9]:
tables = Path.cwd().parent / "results" / "tables"
tables.mkdir(parents=True, exist_ok=True)
out.merge(pw, on=["variable", "level", "n_email_vs_none"]).to_csv(tables / "segment_family_power.csv", index=False)
level_counts.to_csv(tables / "segment_level_arm_counts.csv", index=False)
exp.to_csv(tables / "segment_expected_conversions.csv", index=False)
print("saved results/tables/segment_family_power.csv , results/tables/segment_level_arm_counts.csv, and results/tables/segment_expected_conversions.csv")

saved results/tables/segment_family_power.csv , results/tables/segment_level_arm_counts.csv, and results/tables/segment_expected_conversions.csv
